<a href="https://colab.research.google.com/github/mj87246-ai/kfa-ai-agent/blob/main/mini_financial_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:

import requests
from google.colab import userdata

api_key = userdata.get("DART_API_KEY")

print("API 키 준비 완료:", bool(api_key))


API 키 준비 완료: True


In [25]:
# sk하이닉스 기준
url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"

params = {
    "crtfc_key": api_key,
    "corp_code": "00164779",
    "bsns_year": "2025",
    "reprt_code": "11011",
    "fs_div": "CFS"
}

response = requests.get(url, params=params, timeout=20)

data = response.json()

print(data.get("status"))
print(data.get("message"))


000
정상


In [26]:
records = data["list"]

print("재무제표 데이터 개수:", len(records))
print("첫 번째 데이터:")
print(records[0])

재무제표 데이터 개수: 230
첫 번째 데이터:
{'rcept_no': '20260317000635', 'reprt_code': '11011', 'bsns_year': '2025', 'corp_code': '00164779', 'sj_div': 'BS', 'sj_nm': '재무상태표', 'account_id': 'ifrs-full_Assets', 'account_nm': '자산총계', 'account_detail': '-', 'thstrm_nm': '제 78 기', 'thstrm_amount': '176107659000000', 'frmtrm_nm': '제 77 기', 'frmtrm_amount': '119855209000000', 'bfefrmtrm_nm': '제 76 기', 'bfefrmtrm_amount': '100330165000000', 'ord': '7', 'currency': 'KRW'}


In [27]:
first = records[0]

print("계정명:", first.get("account_nm"))
print("당기 금액:", first.get("thstrm_amount"))
print("재무제표 구분:", first.get("sj_nm"))

계정명: 자산총계
당기 금액: 176107659000000
재무제표 구분: 재무상태표


In [28]:

for record in records:
    account = record["account_nm"]

    if account in ["매출액", "수익(매출액)", "영업이익", "영업이익(손실)"]:
        print("계정명:", account)
        print("계정 ID:", record["account_id"])
        print("재무제표:", record["sj_nm"])
        print("금액:", record["thstrm_amount"])
        print("--------------------")


계정명: 영업이익(손실)
계정 ID: dart_OperatingIncomeLoss
재무제표: 포괄손익계산서
금액: 47206319000000
--------------------
계정명: 매출액
계정 ID: ifrs-full_Revenue
재무제표: 포괄손익계산서
금액: 97146675000000
--------------------


In [29]:
# 숫자 직접 입력
def calculate_op_margin(revenue, operating_income):
    margin = operating_income / revenue * 100
    return margin

revenue = 97146675000000
operating_income = 47206319000000

result = calculate_op_margin(revenue, operating_income)

print(f"영업이익률: {result:.2f}%")


영업이익률: 48.59%


In [30]:
# 숫자 직접 입력
def extract_financials(records):
    revenue = None
    operating_income = None

    for record in records:
        account_id = record["account_id"]

        if account_id == "ifrs-full_Revenue":
            revenue = int(record["thstrm_amount"])

        elif account_id == "dart_OperatingIncomeLoss":
            operating_income = int(record["thstrm_amount"])

    return revenue, operating_income


revenue, operating_income = extract_financials(records)

print("매출액:", revenue)
print("영업이익:", operating_income)


매출액: 97146675000000
영업이익: 47206319000000


In [31]:
# 데이터 수집, 추출, 계산
def get_dart_data(corp_code, year):
    url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"

    params = {
        "crtfc_key": api_key,
        "corp_code": corp_code,
        "bsns_year": str(year),
        "reprt_code": "11011",
        "fs_div": "CFS"
    }

    response = requests.get(url, params=params, timeout=20)
    response.raise_for_status()

    data = response.json()

    if data.get("status") != "000":
        raise ValueError(data.get("message", "DART 조회 실패"))

    return data["list"]


In [32]:

def analyze_company(corp_code, year):
    records = get_dart_data(corp_code, year)

    revenue, operating_income = extract_financials(records)

    if revenue is None or operating_income is None:
        raise ValueError("필요한 재무정보를 찾지 못했습니다.")

    if revenue == 0:
        raise ValueError("매출액이 0이므로 영업이익률을 계산할 수 없습니다.")

    margin = calculate_op_margin(revenue, operating_income)

    result = {
        "corp_code": corp_code,
        "year": year,
        "revenue": revenue,
        "operating_income": operating_income,
        "operating_margin": margin
    }

    return result


In [33]:

company = analyze_company("00164779", 2025)

print("기업 고유번호:", company["corp_code"])
print("사업연도:", company["year"])
print("매출액:", f'{company["revenue"]:,}', "원")
print("영업이익:", f'{company["operating_income"]:,}', "원")
print("영업이익률:", f'{company["operating_margin"]:.2f}', "%")


기업 고유번호: 00164779
사업연도: 2025
매출액: 97,146,675,000,000 원
영업이익: 47,206,319,000,000 원
영업이익률: 48.59 %


In [34]:
# sk하이닉스 2025 기준
def make_common_table(corp_code, year):
    records = get_dart_data(corp_code, year)

    targets = {
        "ifrs-full_Revenue": "매출액",
        "dart_OperatingIncomeLoss": "영업이익"
    }

    rows = []

    for record in records:
        account_id = record["account_id"]

        if account_id in targets:
            raw_amount = record.get("thstrm_amount")
            amount = int(raw_amount) if raw_amount not in (None, "", "-") else None

            rcept_no = record["rcept_no"]
            disclosed_date = (
                f"{rcept_no[:4]}-{rcept_no[4:6]}-{rcept_no[6:8]}"
            )

            row = {
                "종목코드": "000660",
                "종목명": "SK하이닉스",
                "항목": targets[account_id],
                "값": amount,
                "단위": record["currency"],
                "기준": "연결",
                "기준일": f"{year}-12-31",
                "공시일": disclosed_date,
                "출처": "DART 사업보고서",
                "출처 URL": f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={rcept_no}",
                "신뢰도": "상",
                "수집 방법": "공식 API",
                "수집 시각": pd.Timestamp.now(tz="Asia/Seoul").isoformat(),
                "상태": "형식 통과" if amount is not None else "확인 필요",
                "비고": f"원본 계정 ID: {account_id}"
            }

            rows.append(row)

    return pd.DataFrame(rows)


In [35]:

common_table = make_common_table("00164779", 2025)

display(common_table)

print("행 개수:", len(common_table))
print("열 개수:", len(common_table.columns))


,종목코드,종목명,항목,값,단위,기준,기준일,공시일,출처,출처 URL,신뢰도,수집 방법,수집 시각,상태,비고
0,000660,SK하이닉스,영업이익,47206319000000,KRW,연결,2025-12-31,2026-03-17,DART 사업보고서,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,상,공식 API,2026-11-10T11:05:00.630069+09:00,형식 통과,원본 계정 ID: dart_OperatingIncomeLoss
1,000660,SK하이닉스,매출액,97146675000000,KRW,연결,2025-12-31,2026-03-17,DART 사업보고서,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,상,공식 API,2026-11-10T11:05:00.630266+09:00,형식 통과,원본 계정 ID: ifrs-full_Revenue


행 개수: 2
열 개수: 15


In [36]:
# rid 생성 함수
def make_rid(row):
    return "|".join([
        row["종목코드"],
        row["항목"],
        row["기준"],
        row["기준일"],
        row["출처"]
    ])

common_table["rid"] = common_table.apply(make_rid, axis=1)

display(common_table[["항목", "값", "rid"]])


,항목,값,rid
0,영업이익,47206319000000,000660|영업이익|연결|2025-12-31|DART 사업보고서
1,매출액,97146675000000,000660|매출액|연결|2025-12-31|DART 사업보고서


In [37]:

duplicates = common_table["rid"].duplicated()

print("중복 rid 개수:", duplicates.sum())


중복 rid 개수: 0


In [38]:

common_table = common_table.merge(
    sources,
    on=["출처", "출처 URL"],
    how="left"
)

display(common_table[["항목", "값", "rid", "S#"]])


,항목,값,rid,S#
0,영업이익,47206319000000,000660|영업이익|연결|2025-12-31|DART 사업보고서,S1
1,매출액,97146675000000,000660|매출액|연결|2025-12-31|DART 사업보고서,S1


In [39]:

revenue_row = common_table[
    common_table["항목"] == "매출액"
]

income_row = common_table[
    common_table["항목"] == "영업이익"
]

print("매출액 행 개수:", len(revenue_row))
print("영업이익 행 개수:", len(income_row))


매출액 행 개수: 1
영업이익 행 개수: 1


In [40]:

if len(revenue_row) != 1 or len(income_row) != 1:
    raise ValueError("매출액 또는 영업이익이 없거나 중복되었습니다.")

revenue_data = revenue_row.iloc[0]
income_data = income_row.iloc[0]

if (
    revenue_data["기준"] != income_data["기준"]
    or revenue_data["기준일"] != income_data["기준일"]
    or revenue_data["단위"] != income_data["단위"]
):
    raise ValueError("두 재무정보의 기준·날짜·단위가 다릅니다.")

revenue_value = int(revenue_data["값"])
income_value = int(income_data["값"])

if revenue_value == 0:
    raise ValueError("매출액이 0이어서 계산할 수 없습니다.")

margin = income_value / revenue_value * 100

metric_M1 = {
    "metric_id": "M1",
    "name": "영업이익률",
    "value": margin,
    "unit": "%",
    "formula": "영업이익 / 매출액 * 100",
    "input_rids": [
        revenue_data["rid"],
        income_data["rid"]
    ],
    "source_ids": sorted(set([
        revenue_data["S#"],
        income_data["S#"]
    ]))
}

print(metric_M1)


{'metric_id': 'M1', 'name': '영업이익률', 'value': 48.59283037736495, 'unit': '%', 'formula': '영업이익 / 매출액 * 100', 'input_rids': ['000660|매출액|연결|2025-12-31|DART 사업보고서', '000660|영업이익|연결|2025-12-31|DART 사업보고서'], 'source_ids': ['S1']}


In [41]:

print("지표 ID:", metric_M1["metric_id"])
print("지표명:", metric_M1["name"])
print("계산값:", f'{metric_M1["value"]:.2f}%')
print("계산식:", metric_M1["formula"])

print("\n사용된 데이터:")
for rid in metric_M1["input_rids"]:
    print("-", rid)

print("\n출처:", metric_M1["source_ids"])


지표 ID: M1
지표명: 영업이익률
계산값: 48.59%
계산식: 영업이익 / 매출액 * 100

사용된 데이터:
- 000660|매출액|연결|2025-12-31|DART 사업보고서
- 000660|영업이익|연결|2025-12-31|DART 사업보고서

출처: ['S1']


In [42]:
%pip install -q "google-genai>=2.12.1,<3" "google-auth==2.49.0"

In [43]:
from importlib.metadata import version

print("google-genai:", version("google-genai"))
print("google-auth:", version("google-auth"))

google-genai: 2.12.1
google-auth: 2.49.0


In [44]:
from google import genai
from google.colab import userdata

gemini_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=gemini_key)

print("Gemini 키 준비:", bool(gemini_key))

Gemini 키 준비: True


In [45]:
for model in client.models.list():
    if "generateContent" in (model.supported_actions or []):
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
models

In [51]:

import time
from google.genai import errors

model_name = "gemini-3.8-flash"

for attempt in range(3):
    try:
        response = client.models.generate_content(
            model=model_name,
            contents="API 연결 테스트입니다. 성공이라고 답해주세요."
        )

        print("Gemini 응답:", response.text)
        break

    except errors.ServerError as e:
        if e.code != 503:
            raise

        if attempt == 2:
            print("서버 혼잡으로 3회 모두 실패했습니다.")
            break

        delay = 5 * (2 ** attempt)
        print(f"503 오류. {delay}초 후 재시도합니다.")
        time.sleep(delay)


Gemini 응답: 성공


In [52]:

import json

financial_data = {
    "기업": "SK하이닉스",
    "사업연도": 2025,
    "매출액_원": int(revenue_data["값"]),
    "영업이익_원": int(income_data["값"]),
    "영업이익률": round(metric_M1["value"], 4),
    "지표_ID": metric_M1["metric_id"],
    "출처_ID": metric_M1["source_ids"]
}

print(json.dumps(financial_data, ensure_ascii=False, indent=2))


{
  "기업": "SK하이닉스",
  "사업연도": 2025,
  "매출액_원": 97146675000000,
  "영업이익_원": 47206319000000,
  "영업이익률": 48.5928,
  "지표_ID": "M1",
  "출처_ID": [
    "S1"
  ]
}


In [53]:

prompt = f"""
너는 기업 재무분석 보조 AI다.

아래 데이터만 사용하여 한국어로 재무분석 문단을 작성하라.

{json.dumps(financial_data, ensure_ascii=False, indent=2)}

규칙:
1. 매출액과 영업이익을 조 원 단위로 요약하라.
2. 영업이익률을 설명하라.
3. 영업이익률 수치 옆에 [M1]을 표시하라.
4. 매출액과 영업이익 수치 옆에 [S1]을 표시하라.
5. 전년 대비 증가율, 산업 전망, 주가 전망은 추측하지 마라.
6. 제공되지 않은 숫자를 만들지 마라.
7. 마지막에 분석의 한계를 한 문장으로 작성하라.

3~5문장으로 작성하라.
"""


In [54]:

response = client.models.generate_content(
    model=model_name,
    contents=prompt
)

analysis_text = response.text

print(analysis_text)


SK하이닉스의 2025년 사업연도 매출액은 약 97.1조 원[S1]이며, 영업이익은 약 47.2조 원[S1]입니다. 매출액 대비 영업이익의 비율을 나타내는 지표인 영업이익률은 48.5928%[M1]를 기록했습니다. 본 분석은 제공된 2025년 재무 데이터만을 바탕으로 작성되어, 과거 실적과의 비교나 향후 추세를 파악하기에는 한계가 있습니다.


In [55]:

industry_sources = [
    {
        "id": "I1",
        "기관": "WSTS",
        "발표일": "2026-08-13",
        "주제": "세계 반도체 시장",
        "내용": (
            "2026년 상반기 세계 반도체 시장은 "
            "전년 동기 대비 102% 성장했다. "
            "메모리 부문은 305% 성장했다. "
            "AI 인프라 투자가 주요 성장 배경이다."
        ),
        "url": "https://wsts.tsia.org.tw/PageDetail.aspx?pageID=1689"
    },
    {
        "id": "I2",
        "기관": "SK하이닉스",
        "발표일": "2026-01-28",
        "주제": "HBM과 서버용 메모리",
        "내용": (
            "SK하이닉스의 2025년 HBM 매출은 "
            "전년 대비 두 배 이상 증가했다. "
            "서버용 일반 메모리 수요도 증가했다고 발표했다."
        ),
        "url": "https://news.skhynix.com/en/sk-hynix-announces-fy25-financial-results/"
    }
]

for source in industry_sources:
    print(source["id"], source["기관"], source["주제"])


I1 WSTS 세계 반도체 시장
I2 SK하이닉스 HBM과 서버용 메모리


In [56]:

industry_table = pd.DataFrame(industry_sources)

display(industry_table)

print("산업 자료 개수:", len(industry_table))


,id,기관,발표일,주제,내용,url
0,I1,WSTS,2026-08-13,세계 반도체 시장,2026년 상반기 세계 반도체 시장은 전년 동기 대비 102% 성장했다. 메모리 부...,https://wsts.tsia.org.tw/PageDetail.aspx?pageI...
1,I2,SK하이닉스,2026-01-28,HBM과 서버용 메모리,SK하이닉스의 2025년 HBM 매출은 전년 대비 두 배 이상 증가했다. 서버용 일...,https://news.skhynix.com/en/sk-hynix-announces...


산업 자료 개수: 2


In [60]:

import json
import time
import random
from google.genai import errors

def analyze_industry(sources):

    source_text = json.dumps(
        sources,
        ensure_ascii=False,
        indent=2
    )

    prompt = f"""
    너는 메모리 반도체 산업분석 담당 AI다.

    다음 자료만 사용하여 분석하라.

    {source_text}

    [분석 항목]
    1. 반도체 산업의 성장 동력
    2. SK하이닉스에 대한 시사점
    3. 주요 위험 요인과 추가 확인 사항

    [규칙]
    - 사실과 추론을 구분하라.
    - 근거가 있는 주장에는 [I1], [I2]를 표시하라.
    - 자료에 없는 수치를 만들어내지 마라.
    - 확인되지 않은 위험 요인은 가설로 표시하라.
    - 투자 의견과 목표주가는 제시하지 마라.

    한국어로 작성하라.
    """

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt
            )

            if not response.text:
                raise ValueError("Gemini 응답이 비어 있습니다.")

            return response.text

        except errors.ServerError as e:
            if e.code != 503:
                raise

            if attempt == 2:
                raise RuntimeError(
                    "Gemini 서버 혼잡으로 3회 실패했습니다."
                ) from e

            delay = 5 * (2 ** attempt) + random.uniform(0, 2)

            print(f"503 오류. {delay:.1f}초 후 재시도")
            time.sleep(delay)


In [61]:
industry_analysis = analyze_industry(industry_sources)

print(industry_analysis)

제시된 자료를 바탕으로 분석한 메모리 반도체 산업 현황은 다음과 같습니다.

---

### 1. 반도체 산업의 성장 동력

* **사실:**
  * 2026년 상반기 세계 반도체 시장은 전년 동기 대비 102% 성장하였으며, 특히 메모리 부문은 305% 성장했습니다 [I1].
  * 이러한 성장의 핵심 동력은 'AI 인프라 투자'입니다 [I1].
  * AI 인프라뿐만 아니라 서버용 일반 메모리 수요 역시 증가하고 있습니다 [I2].

* **추론:**
  * 전체 반도체 시장 성장률(102%) 대비 메모리 부문 성장률(305%)이 압도적으로 높다는 점에서, 현재 반도체 사이클의 주도권은 AI 인프라 확장에 필수적인 메모리가 쥐고 있는 것으로 판단됩니다 [I1].

---

### 2. SK하이닉스에 대한 시사점

* **사실:**
  * SK하이닉스의 2025년 HBM(고대역폭 메모리) 매출은 전년 대비 두 배 이상 증가했습니다 [I2].
  * 서버용 일반 메모리 부문에서도 수요 증가를 확인했습니다 [I2].

* **추론:**
  * AI 인프라 투자 확대로 촉발된 메모리 시장의 급성장(2026년 상반기 305% 성장) 환경은 HBM 매출이 급증하고 있는 SK하이닉스의 사업 포트폴리오와 직접적으로 맞물려 긍정적인 사업 환경을 조성하고 있습니다 [I1, I2].
  * HBM뿐만 아니라 서버용 일반 메모리 수요도 함께 증가함에 따라, 서버향 메모리 전반에서 실적 기여가 확대되었을 것으로 추론할 수 있습니다 [I2].

---

### 3. 주요 위험 요인과 추가 확인 사항

* **주요 위험 요인 (가설):**
  * **[가설] AI 인프라 투자 둔화 리스크:** 현재 성장의 주된 배경이 AI 인프라 투자에 집중되어 있으므로 [I1], 빅테크 등 전방 산업의 AI 인프라 투자가 둔화되거나 조정기에 진입할 경우 메모리 시장의 성장세가 급격히 꺾일 위험이 있습니다.
  * **[가설] 공급 확대에 따른 수급 불균형 리스크:** 305%에 달하는 메모리 시장

In [62]:

industry_result = {
    "part": "산업분석",
    "company": "SK하이닉스",
    "analysis_date": "2026-11-10",
    "analysis": industry_analysis,
    "source_ids": [s["id"] for s in industry_sources],
    "status": "검토 필요"
}

print("분석 파트:", industry_result["part"])
print("출처:", industry_result["source_ids"])
print("검토 상태:", industry_result["status"])


분석 파트: 산업분석
출처: ['I1', 'I2']
검토 상태: 검토 필요


In [63]:

def get_two_year_financials(records):
    targets = {
        "ifrs-full_Revenue": "매출액",
        "dart_OperatingIncomeLoss": "영업이익"
    }

    result = {}

    for record in records:
        account_id = record["account_id"]

        if account_id in targets:
            name = targets[account_id]

            result[name] = {
                "2024": int(record["frmtrm_amount"]),
                "2025": int(record["thstrm_amount"])
            }

    return result


In [64]:

two_year_data = get_two_year_financials(records)

for name, values in two_year_data.items():
    print(name)
    print("2024년:", f'{values["2024"]:,}', "원")
    print("2025년:", f'{values["2025"]:,}', "원")
    print()


영업이익
2024년: 23,467,319,000,000 원
2025년: 47,206,319,000,000 원

매출액
2024년: 66,192,960,000,000 원
2025년: 97,146,675,000,000 원



In [65]:

def calculate_growth(previous, current):
    if previous <= 0:
        return None

    growth = (current - previous) / previous * 100

    return growth


revenue_growth = calculate_growth(
    two_year_data["매출액"]["2024"],
    two_year_data["매출액"]["2025"]
)

income_growth = calculate_growth(
    two_year_data["영업이익"]["2024"],
    two_year_data["영업이익"]["2025"]
)

print("매출 성장률:", revenue_growth)
print("영업이익 성장률:", income_growth)


매출 성장률: 46.76285061130368
영업이익 성장률: 101.15769935202228


In [66]:

comparison = pd.DataFrame([
    {
        "항목": "매출액",
        "2024년": two_year_data["매출액"]["2024"],
        "2025년": two_year_data["매출액"]["2025"],
        "성장률(%)": revenue_growth
    },
    {
        "항목": "영업이익",
        "2024년": two_year_data["영업이익"]["2024"],
        "2025년": two_year_data["영업이익"]["2025"],
        "성장률(%)": income_growth
    }
])

display(comparison)


,항목,2024년,2025년,성장률(%)
0,매출액,66192960000000,97146675000000,46.762851
1,영업이익,23467319000000,47206319000000,101.157699


In [67]:

targets = [
    "ifrs-full_Assets",
    "ifrs-full_Liabilities",
    "ifrs-full_Equity"
]

for record in records:
    if (
        record["sj_div"] == "BS"
        and record["account_id"] in targets
        and record["account_detail"] == "-"
    ):
        print("계정명:", record["account_nm"])
        print("계정 ID:", record["account_id"])
        print("2025년:", record["thstrm_amount"])
        print("2024년:", record["frmtrm_amount"])
        print("--------------------")


계정명: 자산총계
계정 ID: ifrs-full_Assets
2025년: 176107659000000
2024년: 119855209000000
--------------------
계정명: 자본총계
계정 ID: ifrs-full_Equity
2025년: 120666751000000
2024년: 73915704000000
--------------------
계정명: 부채총계
계정 ID: ifrs-full_Liabilities
2025년: 55440908000000
2024년: 45939505000000
--------------------


In [68]:

def calculate_debt_ratio(records):
    targets = {
        "ifrs-full_Assets": "자산",
        "ifrs-full_Liabilities": "부채",
        "ifrs-full_Equity": "자본"
    }

    values = {}

    for record in records:
        if (
            record["sj_div"] == "BS"
            and record["account_detail"] == "-"
            and record["account_id"] in targets
        ):
            name = targets[record["account_id"]]

            if name in values:
                raise ValueError(f"{name} 계정이 중복되었습니다.")

            values[name] = {
                "2024": int(record["frmtrm_amount"]),
                "2025": int(record["thstrm_amount"])
            }

    if set(values) != {"자산", "부채", "자본"}:
        raise ValueError("필요한 재무상태표 계정이 없습니다.")

    ratios = {}

    for year in ["2024", "2025"]:
        assets = values["자산"][year]
        liabilities = values["부채"][year]
        equity = values["자본"][year]

        if assets != liabilities + equity:
            raise ValueError(f"{year}년 회계등식 불일치")

        if equity <= 0:
            raise ValueError(f"{year}년 자본이 0 이하입니다.")

        ratios[year] = liabilities / equity * 100

    return ratios


In [69]:

debt_ratios = calculate_debt_ratio(records)

print(f"2024년 부채비율: {debt_ratios['2024']:.2f}%")
print(f"2025년 부채비율: {debt_ratios['2025']:.2f}%")

change = debt_ratios["2025"] - debt_ratios["2024"]

print(f"부채비율 변화: {change:.2f}%p")


2024년 부채비율: 62.15%
2025년 부채비율: 45.95%
부채비율 변화: -16.21%p


In [70]:

keywords = [
    "ProfitLoss",
    "EquityAttributableToOwnersOfParent",
    "CashFlowsFromUsedInOperatingActivities"
]

for record in records:
    account_id = record["account_id"]

    if any(word in account_id for word in keywords):
        print("재무제표:", record["sj_nm"])
        print("계정명:", record["account_nm"])
        print("계정 ID:", account_id)
        print("계정 상세:", record["account_detail"])
        print("2025년:", record["thstrm_amount"])
        print("2024년:", record["frmtrm_amount"])
        print("--------------------")


재무제표: 재무상태표
계정명: 지배기업의 소유지분
계정 ID: ifrs-full_EquityAttributableToOwnersOfParent
계정 상세: -
2025년: 120516178000000
2024년: 73903394000000
--------------------
재무제표: 포괄손익계산서
계정명: 당기순이익(손실)
계정 ID: ifrs-full_ProfitLoss
계정 상세: -
2025년: 42947902000000
2024년: 19796902000000
--------------------
재무제표: 포괄손익계산서
계정명: 비지배지분
계정 ID: ifrs-full_ProfitLossAttributableToNoncontrollingInterests
계정 상세: -
2025년: 28615000000
2024년: 8221000000
--------------------
재무제표: 포괄손익계산서
계정명: 지배기업의 소유주지분
계정 ID: ifrs-full_ProfitLossAttributableToOwnersOfParent
계정 상세: -
2025년: 42919287000000
2024년: 19788681000000
--------------------
재무제표: 포괄손익계산서
계정명: 법인세비용차감전순이익(손실)
계정 ID: ifrs-full_ProfitLossBeforeTax
계정 상세: -
2025년: 50465552000000
2024년: 23885350000000
--------------------
재무제표: 포괄손익계산서
계정명: 지분법투자 관련 손익
계정 ID: ifrs-full_ShareOfProfitLossOfAssociatesAndJointVenturesAccountedForUsingEquityMethod
계정 상세: -
2025년: -564553000000
2024년: -38245000000
--------------------
재무제표: 현금흐름표
계정명: 영업활동 현금흐름
계정 ID: ifrs-full_CashFlowsFro

In [71]:

def find_account(records, account_id, sj_div):
    matches = []

    for record in records:
        if (
            record["account_id"] == account_id
            and record["sj_div"] == sj_div
            and record["account_detail"] == "-"
        ):
            matches.append(record)

    if len(matches) != 1:
        raise ValueError(
            f"{account_id}: 찾은 행이 {len(matches)}개입니다."
        )

    return matches[0]


In [72]:

def calculate_roe(records):
    profit = find_account(
        records,
        "ifrs-full_ProfitLossAttributableToOwnersOfParent",
        "CIS"
    )

    equity = find_account(
        records,
        "ifrs-full_EquityAttributableToOwnersOfParent",
        "BS"
    )

    net_income = int(profit["thstrm_amount"])

    equity_2024 = int(equity["frmtrm_amount"])
    equity_2025 = int(equity["thstrm_amount"])

    if profit["currency"] != equity["currency"]:
        raise ValueError("통화 단위가 서로 다릅니다.")

    average_equity = (equity_2024 + equity_2025) / 2

    if average_equity <= 0:
        raise ValueError("평균 자기자본이 0 이하입니다.")

    roe = net_income / average_equity * 100

    return roe


In [73]:

def calculate_cfo_growth(records):
    cashflow = find_account(
        records,
        "ifrs-full_CashFlowsFromUsedInOperatingActivities",
        "CF"
    )

    cfo_2024 = int(cashflow["frmtrm_amount"])
    cfo_2025 = int(cashflow["thstrm_amount"])

    if cfo_2024 <= 0:
        raise ValueError(
            "전기 영업현금흐름이 0 이하이므로 일반 성장률 계산을 보류합니다."
        )

    growth = (cfo_2025 - cfo_2024) / cfo_2024 * 100

    return {
        "2024": cfo_2024,
        "2025": cfo_2025,
        "growth": growth
    }


In [74]:

roe_2025 = calculate_roe(records)

cfo_result = calculate_cfo_growth(records)

print(f"2025년 ROE: {roe_2025:.2f}%")
print(f"2024년 영업현금흐름: {cfo_result['2024']:,}원")
print(f"2025년 영업현금흐름: {cfo_result['2025']:,}원")
print(f"영업현금흐름 증가율: {cfo_result['growth']:.2f}%")


2025년 ROE: 44.15%
2024년 영업현금흐름: 29,795,885,000,000원
2025년 영업현금흐름: 53,373,126,000,000원
영업현금흐름 증가율: 79.13%


In [75]:

financial_metrics = {
    "M1": {
        "name": "영업이익률",
        "value": metric_M1["value"],
        "unit": "%",
        "formula": "영업이익 / 매출액 * 100"
    },
    "M2": {
        "name": "매출 성장률",
        "value": revenue_growth,
        "unit": "%",
        "formula": "(당기 매출 - 전기 매출) / 전기 매출 * 100"
    },
    "M3": {
        "name": "영업이익 성장률",
        "value": income_growth,
        "unit": "%",
        "formula": "(당기 영업이익 - 전기 영업이익) / 전기 영업이익 * 100"
    },
    "M4": {
        "name": "부채비율",
        "value": debt_ratios["2025"],
        "unit": "%",
        "formula": "부채총계 / 자본총계 * 100"
    },
    "M5": {
        "name": "ROE",
        "value": roe_2025,
        "unit": "%",
        "formula": "지배기업 귀속 순이익 / 평균 지배기업 귀속 자본 * 100"
    },
    "M6": {
        "name": "영업현금흐름 성장률",
        "value": cfo_result["growth"],
        "unit": "%",
        "formula": "(당기 현금흐름 - 전기 현금흐름) / 전기 현금흐름 * 100"
    }
}

for metric_id, metric in financial_metrics.items():
    print(
        metric_id,
        metric["name"],
        f'{metric["value"]:.2f}{metric["unit"]}'
    )


M1 영업이익률 48.59%
M2 매출 성장률 46.76%
M3 영업이익 성장률 101.16%
M4 부채비율 45.95%
M5 ROE 44.15%
M6 영업현금흐름 성장률 79.13%


In [77]:

import math

for metric_id, metric in financial_metrics.items():
    value = metric["value"]

    if value is None:
        raise ValueError(f"{metric_id}: 계산값이 없습니다.")

    if not math.isfinite(float(value)):
        raise ValueError(f"{metric_id}: 유효하지 않은 숫자입니다.")

print("6개 지표의 숫자 형식 검사 통과")


6개 지표의 숫자 형식 검사 통과


In [78]:

account_targets = {
    "매출액": ("ifrs-full_Revenue", "CIS"),
    "영업이익": ("dart_OperatingIncomeLoss", "CIS"),
    "부채총계": ("ifrs-full_Liabilities", "BS"),
    "자본총계": ("ifrs-full_Equity", "BS"),
    "지배기업 귀속 자본": (
        "ifrs-full_EquityAttributableToOwnersOfParent", "BS"
    ),
    "지배기업 귀속 순이익": (
        "ifrs-full_ProfitLossAttributableToOwnersOfParent", "CIS"
    ),
    "영업활동현금흐름": (
        "ifrs-full_CashFlowsFromUsedInOperatingActivities", "CF"
    )
}

print("필요한 계정 수:", len(account_targets))


필요한 계정 수: 7


In [79]:

evidence_rows = []

for name, (account_id, sj_div) in account_targets.items():

    record = find_account(records, account_id, sj_div)

    for year, amount_key in [
        (2024, "frmtrm_amount"),
        (2025, "thstrm_amount")
    ]:
        value = int(record[amount_key])

        evidence_rows.append({
            "기업": "SK하이닉스",
            "항목": name,
            "연도": year,
            "값": value,
            "단위": record["currency"],
            "계정ID": account_id,
            "재무제표": sj_div,
            "원본 공시번호": record["rcept_no"],
            "source_id": "S1"
        })

evidence_table = pd.DataFrame(evidence_rows)

display(evidence_table[
    ["항목", "연도", "값", "source_id"]
])

print("근거 데이터 개수:", len(evidence_table))


,항목,연도,값,source_id
0,매출액,2024,66192960000000,S1
1,매출액,2025,97146675000000,S1
2,영업이익,2024,23467319000000,S1
3,영업이익,2025,47206319000000,S1
4,부채총계,2024,45939505000000,S1
5,부채총계,2025,55440908000000,S1
6,자본총계,2024,73915704000000,S1
7,자본총계,2025,120666751000000,S1
8,지배기업 귀속 자본,2024,73903394000000,S1
9,지배기업 귀속 자본,2025,120516178000000,S1


근거 데이터 개수: 14


In [80]:

evidence_table["rid"] = (
    evidence_table["기업"] + "|" +
    evidence_table["계정ID"] + "|" +
    evidence_table["재무제표"] + "|" +
    evidence_table["연도"].astype(str) + "|" +
    evidence_table["원본 공시번호"]
)

if evidence_table["rid"].duplicated().any():
    raise ValueError("중복된 rid가 있습니다.")

display(evidence_table[
    ["항목", "연도", "rid"]
])


,항목,연도,rid
0,매출액,2024,SK하이닉스|ifrs-full_Revenue|CIS|2024|20260317000635
1,매출액,2025,SK하이닉스|ifrs-full_Revenue|CIS|2025|20260317000635
2,영업이익,2024,SK하이닉스|dart_OperatingIncomeLoss|CIS|2024|20260...
3,영업이익,2025,SK하이닉스|dart_OperatingIncomeLoss|CIS|2025|20260...
4,부채총계,2024,SK하이닉스|ifrs-full_Liabilities|BS|2024|202603170...
5,부채총계,2025,SK하이닉스|ifrs-full_Liabilities|BS|2025|202603170...
6,자본총계,2024,SK하이닉스|ifrs-full_Equity|BS|2024|20260317000635
7,자본총계,2025,SK하이닉스|ifrs-full_Equity|BS|2025|20260317000635
8,지배기업 귀속 자본,2024,SK하이닉스|ifrs-full_EquityAttributableToOwnersOfP...
9,지배기업 귀속 자본,2025,SK하이닉스|ifrs-full_EquityAttributableToOwnersOfP...


In [81]:

def find_rid(item, year):
    matched = evidence_table[
        (evidence_table["항목"] == item)
        & (evidence_table["연도"] == year)
    ]

    if len(matched) != 1:
        raise ValueError(
            f"{item} {year}년 데이터가 "
            f"{len(matched)}개 발견됐습니다."
        )

    return matched.iloc[0]["rid"]


In [82]:

metric_inputs = {
    "M1": [
        ("영업이익", 2025),
        ("매출액", 2025)
    ],
    "M2": [
        ("매출액", 2024),
        ("매출액", 2025)
    ],
    "M3": [
        ("영업이익", 2024),
        ("영업이익", 2025)
    ],
    "M4": [
        ("부채총계", 2025),
        ("자본총계", 2025)
    ],
    "M5": [
        ("지배기업 귀속 순이익", 2025),
        ("지배기업 귀속 자본", 2024),
        ("지배기업 귀속 자본", 2025)
    ],
    "M6": [
        ("영업활동현금흐름", 2024),
        ("영업활동현금흐름", 2025)
    ]
}


In [83]:

metric_inputs = {
    "M1": [
        ("영업이익", 2025),
        ("매출액", 2025)
    ],
    "M2": [
        ("매출액", 2024),
        ("매출액", 2025)
    ],
    "M3": [
        ("영업이익", 2024),
        ("영업이익", 2025)
    ],
    "M4": [
        ("부채총계", 2025),
        ("자본총계", 2025)
    ],
    "M5": [
        ("지배기업 귀속 순이익", 2025),
        ("지배기업 귀속 자본", 2024),
        ("지배기업 귀속 자본", 2025)
    ],
    "M6": [
        ("영업활동현금흐름", 2024),
        ("영업활동현금흐름", 2025)
    ]
}


In [85]:

for metric_id, inputs in metric_inputs.items():

    rids = [
        find_rid(item, year)
        for item, year in inputs
    ]

    matched_rows = evidence_table[
        evidence_table["rid"].isin(rids)
    ]

    if len(matched_rows) != len(rids):
        raise ValueError(
            f"{metric_id}: 근거 데이터 연결 실패"
        )

    financial_metrics[metric_id]["input_rids"] = rids

    financial_metrics[metric_id]["source_ids"] = sorted(
        matched_rows["source_id"].unique().tolist()
    )

print("6개 지표의 근거 연결 완료")


6개 지표의 근거 연결 완료


In [86]:

for metric_id, metric in financial_metrics.items():
    print(f"\n[{metric_id}] {metric['name']}")
    print(f"계산값: {metric['value']:.2f}{metric['unit']}")
    print(f"원본 데이터 수: {len(metric['input_rids'])}")
    print(f"출처: {metric['source_ids']}")



[M1] 영업이익률
계산값: 48.59%
원본 데이터 수: 2
출처: ['S1']

[M2] 매출 성장률
계산값: 46.76%
원본 데이터 수: 2
출처: ['S1']

[M3] 영업이익 성장률
계산값: 101.16%
원본 데이터 수: 2
출처: ['S1']

[M4] 부채비율
계산값: 45.95%
원본 데이터 수: 2
출처: ['S1']

[M5] ROE
계산값: 44.15%
원본 데이터 수: 3
출처: ['S1']

[M6] 영업현금흐름 성장률
계산값: 79.13%
원본 데이터 수: 2
출처: ['S1']


In [87]:

import json

metric_payload = []

for metric_id, metric in financial_metrics.items():
    metric_payload.append({
        "id": metric_id,
        "name": metric["name"],
        "value": round(metric["value"], 4),
        "unit": metric["unit"],
        "formula": metric["formula"],
        "input_rids": metric["input_rids"],
        "source_ids": metric["source_ids"]
    })

print(json.dumps(metric_payload, ensure_ascii=False, indent=2))


[
  {
    "id": "M1",
    "name": "영업이익률",
    "value": 48.5928,
    "unit": "%",
    "formula": "영업이익 / 매출액 * 100",
    "input_rids": [
      "SK하이닉스|dart_OperatingIncomeLoss|CIS|2025|20260317000635",
      "SK하이닉스|ifrs-full_Revenue|CIS|2025|20260317000635"
    ],
    "source_ids": [
      "S1"
    ]
  },
  {
    "id": "M2",
    "name": "매출 성장률",
    "value": 46.7629,
    "unit": "%",
    "formula": "(당기 매출 - 전기 매출) / 전기 매출 * 100",
    "input_rids": [
      "SK하이닉스|ifrs-full_Revenue|CIS|2024|20260317000635",
      "SK하이닉스|ifrs-full_Revenue|CIS|2025|20260317000635"
    ],
    "source_ids": [
      "S1"
    ]
  },
  {
    "id": "M3",
    "name": "영업이익 성장률",
    "value": 101.1577,
    "unit": "%",
    "formula": "(당기 영업이익 - 전기 영업이익) / 전기 영업이익 * 100",
    "input_rids": [
      "SK하이닉스|dart_OperatingIncomeLoss|CIS|2024|20260317000635",
      "SK하이닉스|dart_OperatingIncomeLoss|CIS|2025|20260317000635"
    ],
    "source_ids": [
      "S1"
    ]
  },
  {
    "id": "M4",
    "name": "부채비율",
   

In [90]:

import time
from google.genai import errors

def analyze_financials(metrics):
    prompt = f"""
너는 SK하이닉스 기업·재무분석 담당 AI다.

다음 지표는 2025년 사업보고서를 바탕으로
Python 코드가 계산한 결과다.

{json.dumps(metrics, ensure_ascii=False, indent=2)}

[분석 항목]
1. 성장성: 매출 및 영업이익 증가율
2. 수익성: 영업이익률 및 ROE
3. 재무안정성: 부채비율
4. 현금흐름: 영업활동현금흐름 증가율
5. 종합 평가와 추가 확인 사항

[규칙]
- 제공된 수치만 사용하라.
- 모든 수치 주장 뒤에 해당 [M#]를 표시하라.
- 수치에서 확인되는 사실과 추론을 구분하라.
- 업종 평균과 비교하지 마라. 비교 데이터가 없다.
- ROE가 높다는 이유만으로 우량 기업이라고 단정하지 마라.
- 현금흐름 증가만으로 이익의 질을 확정하지 마라.
- 제공되지 않은 사업 경쟁력이나 주가 전망을 추정하지 마라.
- 산업분석 결과는 이번 입력에 없으므로 사용하지 마라.

한국어로 5개 소제목을 나누어 작성하라.
"""

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt
            )

            if not response.text:
                raise ValueError("Gemini 응답이 비어 있습니다.")

            return response.text

        except errors.ServerError as e:
            if e.code != 503:
                raise
            if attempt == 2:
                raise RuntimeError(
                    "Gemini 서버 혼잡으로 분석에 실패했습니다."
                ) from e

            delay = 5 * (2 ** attempt)
            print(f"503 오류. {delay}초 후 재시도")
            time.sleep(delay)


In [91]:

financial_analysis = analyze_financials(metric_payload)

print(financial_analysis)


SK하이닉스의 2025년 사업보고서 기반 재무지표 분석 보고서입니다.

---

### 1. 성장성: 매출 및 영업이익 증가율

* **확인된 사실**: 
  * 2025년 매출 성장률은 46.7629%를 기록했습니다 [M2].
  * 2025년 영업이익 성장률은 101.1577%로 집계되었습니다 [M3].
* **추론 및 분석**: 
  * 매출 성장률[M2] 대비 영업이익 성장률[M3]이 2배 이상 높게 나타났습니다. 이는 전년 대비 매출 증가 폭보다 영업이익의 증가 폭이 더 크게 확대되었음을 의미합니다.

### 2. 수익성: 영업이익률 및 ROE

* **확인된 사실**: 
  * 2025년 영업이익률은 48.5928%입니다 [M1].
  * 지배기업 귀속 순이익과 평균 지배기업 귀속 자본을 바탕으로 계산된 ROE는 44.1512%입니다 [M5].
* **추론 및 분석**: 
  * 매출액의 절반에 가까운 48.5928%가 영업이익으로 남는 높은 수익성을 보였습니다 [M1]. 
  * ROE가 44.1512%로 높은 수치를 나타내고 있으나 [M5], 이는 당기순이익 규모와 자기자본 변동의 결과값이며, 이 수치만으로 기업 전체의 장기적 우량성을 단정할 수는 없습니다.

### 3. 재무안정성: 부채비율

* **확인된 사실**: 
  * 2025년 기준 부채비율(부채총계 / 자본총계)은 45.9455%입니다 [M4].
* **추론 및 분석**: 
  * 자본총계 대비 부채총계의 비중이 50% 미만(45.9455%)을 유지하고 있어 [M4], 타인자본보다 자기자본의 비중이 더 큰 구조를 형성하고 있음을 확인할 수 있습니다.

### 4. 현금흐름: 영업활동현금흐름 증가율

* **확인된 사실**: 
  * 2025년 영업활동현금흐름 성장률은 전기 대비 79.1292% 증가했습니다 [M6].
* **추론 및 분석**: 
  * 영업활동을 통해 유입된 현금흐름이 전년 대비 79.1292% 큰 폭으로 증가했습니다 [M6]. 다만, 영업현금흐름 증가율 수치 하나만

In [92]:

financial_result = {
    "part": "기업분석",
    "company": "SK하이닉스",
    "year": 2025,
    "analysis": financial_analysis,
    "metrics": financial_metrics,
    "status": "검토 필요"
}

print("분석 파트:", financial_result["part"])
print("지표 개수:", len(financial_result["metrics"]))
print("검토 상태:", financial_result["status"])


분석 파트: 기업분석
지표 개수: 6
검토 상태: 검토 필요


In [93]:

for record in records:
    if (
        record["sj_div"] == "CIS"
        and record["account_detail"] == "-"
        and "EarningsLossPerShare" in record["account_id"]
    ):
        print("계정명:", record["account_nm"])
        print("계정 ID:", record["account_id"])
        print("2025년:", record["thstrm_amount"])
        print("2024년:", record["frmtrm_amount"])
        print("단위:", record["currency"])
        print("--------------------")


계정명: 기본주당순이익(손실)
계정 ID: ifrs-full_BasicEarningsLossPerShare
2025년: 62044
2024년: 28732
단위: KRW
--------------------
계정명: 희석주당순이익(손실)
계정 ID: ifrs-full_DilutedEarningsLossPerShare
2025년: 60378
2024년: 28419
단위: KRW
--------------------


In [94]:

basic_eps_record = find_account(
    records,
    "ifrs-full_BasicEarningsLossPerShare",
    "CIS"
)

diluted_eps_record = find_account(
    records,
    "ifrs-full_DilutedEarningsLossPerShare",
    "CIS"
)

basic_eps = int(basic_eps_record["thstrm_amount"])
diluted_eps = int(diluted_eps_record["thstrm_amount"])

print("기본 EPS:", f"{basic_eps:,}원/주")
print("희석 EPS:", f"{diluted_eps:,}원/주")


기본 EPS: 62,044원/주
희석 EPS: 60,378원/주


In [95]:

def calculate_per_valuation(eps, per_list):
    if eps <= 0:
        raise ValueError("EPS가 0 이하입니다.")

    results = []

    for per in per_list:
        if per <= 0:
            raise ValueError("PER은 양수여야 합니다.")

        price = eps * per

        results.append({
            "EPS": eps,
            "가정 PER": per,
            "시나리오 주가": price
        })

    return pd.DataFrame(results)


In [96]:

per_scenarios = [5, 7, 9]

valuation_table = calculate_per_valuation(
    basic_eps,
    per_scenarios
)

display(valuation_table)


,EPS,가정 PER,시나리오 주가
0,62044,5,310220
1,62044,7,434308
2,62044,9,558396


In [97]:

valuation_scenarios = {
    "비관": {
        "eps_growth": -0.30,
        "per": 5
    },
    "기준": {
        "eps_growth": 0.00,
        "per": 7
    },
    "낙관": {
        "eps_growth": 0.30,
        "per": 9
    }
}

print(valuation_scenarios)


{'비관': {'eps_growth': -0.3, 'per': 5}, '기준': {'eps_growth': 0.0, 'per': 7}, '낙관': {'eps_growth': 0.3, 'per': 9}}


In [98]:

import math

def run_valuation_scenarios(base_eps, scenarios):

    if not math.isfinite(base_eps) or base_eps <= 0:
        raise ValueError("기준 EPS가 유효하지 않습니다.")

    results = []

    for name, assumptions in scenarios.items():

        growth = assumptions["eps_growth"]
        per = assumptions["per"]

        if not math.isfinite(growth) or growth <= -1:
            raise ValueError(f"{name}: EPS 변화율이 유효하지 않습니다.")

        if not math.isfinite(per) or per <= 0:
            raise ValueError(f"{name}: PER이 유효하지 않습니다.")

        expected_eps = base_eps * (1 + growth)
        scenario_price = expected_eps * per

        results.append({
            "시나리오": name,
            "기준 EPS": base_eps,
            "EPS 변화율(%)": growth * 100,
            "예상 EPS": expected_eps,
            "가정 PER": per,
            "시나리오 주가": scenario_price
        })

    return pd.DataFrame(results)


In [99]:

scenario_table = run_valuation_scenarios(
    basic_eps,
    valuation_scenarios
)

display(scenario_table)


,시나리오,기준 EPS,EPS 변화율(%),예상 EPS,가정 PER,시나리오 주가
0,비관,62044,-30.0,43430.8,5,217154.0
1,기준,62044,0.0,62044.0,7,434308.0
2,낙관,62044,30.0,80657.2,9,725914.8


In [100]:

valuation_result = {
    "part": "밸류에이션",
    "company": "SK하이닉스",
    "base_year": 2025,
    "base_eps": basic_eps,
    "eps_type": "기본주당순이익",
    "scenarios": scenario_table.to_dict("records"),
    "assumption_status": "학습용 임의 가정",
    "status": "검토 필요"
}

print("분석 파트:", valuation_result["part"])
print("시나리오 개수:", len(valuation_result["scenarios"]))
print("가정 상태:", valuation_result["assumption_status"])


분석 파트: 밸류에이션
시나리오 개수: 3
가정 상태: 학습용 임의 가정


In [101]:

def get_evidence_value(item, year):
    matched = evidence_table[
        (evidence_table["항목"] == item)
        & (evidence_table["연도"] == year)
    ]

    if len(matched) != 1:
        raise ValueError(
            f"{item}, {year}년: 데이터 {len(matched)}개"
        )

    return int(matched.iloc[0]["값"])


print("2025년 매출액:", get_evidence_value("매출액", 2025))
print("2024년 매출액:", get_evidence_value("매출액", 2024))


2025년 매출액: 97146675000000
2024년 매출액: 66192960000000


In [102]:

def recalculate_metrics():

    r24 = get_evidence_value("매출액", 2024)
    r25 = get_evidence_value("매출액", 2025)

    op24 = get_evidence_value("영업이익", 2024)
    op25 = get_evidence_value("영업이익", 2025)

    debt = get_evidence_value("부채총계", 2025)
    equity = get_evidence_value("자본총계", 2025)

    profit = get_evidence_value("지배기업 귀속 순이익", 2025)
    eq24 = get_evidence_value("지배기업 귀속 자본", 2024)
    eq25 = get_evidence_value("지배기업 귀속 자본", 2025)

    cfo24 = get_evidence_value("영업활동현금흐름", 2024)
    cfo25 = get_evidence_value("영업활동현금흐름", 2025)

    if min(r24, r25, op24, equity, eq24, eq25, cfo24) <= 0:
        raise ValueError("계산에 필요한 양수 조건을 확인하세요.")

    return {
        "M1": op25 / r25 * 100,
        "M2": (r25 - r24) / r24 * 100,
        "M3": (op25 - op24) / op24 * 100,
        "M4": debt / equity * 100,
        "M5": profit / ((eq24 + eq25) / 2) * 100,
        "M6": (cfo25 - cfo24) / cfo24 * 100
    }


recalculated = recalculate_metrics()

print(recalculated)


{'M1': 48.59283037736495, 'M2': 46.76285061130368, 'M3': 101.15769935202228, 'M4': 45.94547175634156, 'M5': 44.15119996252229, 'M6': 79.12918512069703}


In [103]:

import math

validation_results = []

for metric_id, metric in financial_metrics.items():

    original = float(metric["value"])
    checked = float(recalculated[metric_id])

    is_pass = math.isclose(
        original,
        checked,
        rel_tol=1e-10,
        abs_tol=1e-8
    )

    validation_results.append({
        "지표": metric_id,
        "기존 계산값": original,
        "재계산값": checked,
        "판정": "PASS" if is_pass else "FIX"
    })

validation_table = pd.DataFrame(validation_results)

display(validation_table)


,지표,기존 계산값,재계산값,판정
0,M1,48.592830,48.592830,PASS
1,M2,46.762851,46.762851,PASS
2,M3,101.157699,101.157699,PASS
3,M4,45.945472,45.945472,PASS
4,M5,44.151200,44.151200,PASS
5,M6,79.129185,79.129185,PASS


In [104]:

pass_count = (validation_table["판정"] == "PASS").sum()
fix_count = (validation_table["판정"] == "FIX").sum()

print("PASS:", pass_count)
print("FIX:", fix_count)

if fix_count == 0:
    print("6개 재무지표 계산 검증 통과")
else:
    print("재검토가 필요한 지표가 있습니다.")


PASS: 6
FIX: 0
6개 재무지표 계산 검증 통과


In [105]:

def validate_metric_sources(metrics, inputs_map):
    results = []

    all_rids = set(evidence_table["rid"])

    for metric_id, metric in metrics.items():

        expected_rids = [
            find_rid(item, year)
            for item, year in inputs_map[metric_id]
        ]

        actual_rids = metric["input_rids"]

        rid_match = (
            len(actual_rids) == len(expected_rids)
            and set(actual_rids) == set(expected_rids)
            and all(rid in all_rids for rid in actual_rids)
        )

        source_rows = evidence_table[
            evidence_table["rid"].isin(actual_rids)
        ]

        expected_sources = set(source_rows["source_id"])

        source_match = (
            set(metric["source_ids"]) == expected_sources
            and len(source_rows) == len(actual_rids)
        )

        results.append({
            "지표": metric_id,
            "RID검사": "PASS" if rid_match else "FIX",
            "출처검사": "PASS" if source_match else "FIX"
        })

    return pd.DataFrame(results)


source_validation = validate_metric_sources(
    financial_metrics,
    metric_inputs
)

display(source_validation)


,지표,RID검사,출처검사
0,M1,PASS,PASS
1,M2,PASS,PASS
2,M3,PASS,PASS
3,M4,PASS,PASS
4,M5,PASS,PASS
5,M6,PASS,PASS


In [106]:

import re

def validate_report_references(report, metrics):

    cited_ids = re.findall(r"\[(M\d+)\]", report)

    valid_ids = set(metrics.keys())
    cited_set = set(cited_ids)

    invalid_ids = cited_set - valid_ids
    missing_ids = valid_ids - cited_set

    return {
        "used_ids": sorted(cited_set),
        "invalid_ids": sorted(invalid_ids),
        "missing_ids": sorted(missing_ids),
        "status": "PASS" if not invalid_ids else "FIX"
    }


report_check = validate_report_references(
    financial_analysis,
    financial_metrics
)

print("사용된 지표:", report_check["used_ids"])
print("존재하지 않는 지표:", report_check["invalid_ids"])
print("한 번도 인용되지 않은 지표:", report_check["missing_ids"])
print("판정:", report_check["status"])


사용된 지표: ['M1', 'M2', 'M3', 'M4', 'M5', 'M6']
존재하지 않는 지표: []
한 번도 인용되지 않은 지표: []
판정: PASS


In [107]:

import re
import math

def validate_report_numbers(report, metrics):
    results = []

    for line in report.splitlines():

        numbers = re.findall(
            r"(?<!\w)-?\d+(?:,\d{3})*(?:\.\d+)?\s*%",
            line
        )

        ids = re.findall(r"\[(M\d+)\]", line)
        unique_ids = list(dict.fromkeys(ids))

        if not numbers:
            continue

        if len(numbers) != 1 or len(unique_ids) != 1:
            status = "REVIEW"
            reason = "숫자와 지표가 일대일로 대응하지 않음"

        elif unique_ids[0] not in metrics:
            status = "FIX"
            reason = "존재하지 않는 지표 ID"

        else:
            metric_id = unique_ids[0]
            reported = float(
                numbers[0].replace("%", "").replace(",", "")
            )
            expected = float(metrics[metric_id]["value"])

            if math.isclose(
                reported, expected, abs_tol=0.001
            ):
                status = "PASS"
                reason = "계산값과 일치"
            else:
                status = "FIX"
                reason = "계산값과 불일치"

        results.append({
            "문장": line.strip(),
            "판정": status,
            "사유": reason
        })

    return pd.DataFrame(results)


In [108]:

number_validation = validate_report_numbers(
    financial_analysis,
    financial_metrics
)

display(number_validation)

print(
    number_validation["판정"]
    .value_counts()
)


,문장,판정,사유
0,* 2025년 매출 성장률은 46.7629%를 기록했습니다 [M2].,PASS,계산값과 일치
1,* 2025년 영업이익 성장률은 101.1577%로 집계되었습니다 [M3].,PASS,계산값과 일치
2,* 2025년 영업이익률은 48.5928%입니다 [M1].,PASS,계산값과 일치
3,* 지배기업 귀속 순이익과 평균 지배기업 귀속 자본을 바탕으로 계산된 ROE는 44...,PASS,계산값과 일치
4,* 매출액의 절반에 가까운 48.5928%가 영업이익으로 남는 높은 수익성을 보였습...,PASS,계산값과 일치
5,"* ROE가 44.1512%로 높은 수치를 나타내고 있으나 [M5], 이는 당기순이...",PASS,계산값과 일치
6,* 2025년 기준 부채비율(부채총계 / 자본총계)은 45.9455%입니다 [M4].,PASS,계산값과 일치
7,* 자본총계 대비 부채총계의 비중이 50% 미만(45.9455%)을 유지하고 있어 ...,REVIEW,숫자와 지표가 일대일로 대응하지 않음
8,* 2025년 영업활동현금흐름 성장률은 전기 대비 79.1292% 증가했습니다 [M6].,PASS,계산값과 일치
9,* 영업활동을 통해 유입된 현금흐름이 전년 대비 79.1292% 큰 폭으로 증가했습...,PASS,계산값과 일치


판정
PASS      11
REVIEW     4
Name: count, dtype: int64


In [109]:

test_report = """
SK하이닉스의 영업이익률은 48.5928%입니다. [M1]
SK하이닉스의 ROE는 79.13%입니다. [M5]
부채비율은 45.9455%입니다. [M4]
"""

test_result = validate_report_numbers(
    test_report,
    financial_metrics
)

display(test_result)


,문장,판정,사유
0,SK하이닉스의 영업이익률은 48.5928%입니다. [M1],PASS,계산값과 일치
1,SK하이닉스의 ROE는 79.13%입니다. [M5],FIX,계산값과 불일치
2,부채비율은 45.9455%입니다. [M4],PASS,계산값과 일치


In [110]:

import json
import time
from google.genai import errors

def review_financial_logic(report, metrics):

    prompt = f"""
너는 금융 리서치 보고서의 논리 검증 담당자다.

[검증 대상 보고서]
{report}

[검증 근거: 계산 지표]
{json.dumps(metrics, ensure_ascii=False, indent=2)}

다음 항목을 검사하라.

1. 제공된 지표로 뒷받침되지 않는 주장
2. 상관관계를 인과관계로 단정한 주장
3. 전년 비교 데이터 없이 개선됐다고 주장한 부분
4. 사실과 추론을 혼동한 부분
5. 추가 자료가 필요한데 확정적으로 표현한 부분

[검증 규칙]
- 실제 문제가 있는 문장만 지적한다.
- 문제가 없으면 억지로 오류를 만들어내지 않는다.
- 확인할 수 없는 주장은 REVIEW로 분류한다.
- 명백한 수치 오류는 FIX로 분류한다.
- 확인되지 않은 문제를 확정적으로 표현하지 않는다.
- 제공된 자료 밖의 사실을 만들어내지 않는다.
- 검증 대상의 원문과 수정 제안을 구분한다.

다음 JSON 형식으로만 응답하라.

{{
  "issues": [
    {{
      "original": "문제가 있는 원문",
      "status": "REVIEW 또는 FIX",
      "reason": "문제인 이유",
      "suggestion": "수정 제안"
    }}
  ],
  "overall": "PASS 또는 REVIEW 또는 FIX"
}}

문제가 없다면 issues는 빈 배열로 작성하라.
"""

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt
            )

            if not response.text:
                raise ValueError("Gemini 응답이 비어 있습니다.")

            result_text = response.text.strip()

            if result_text.startswith("```"):
                result_text = result_text.split("\n", 1)[1]
                result_text = result_text.rsplit("```", 1)[0].strip()

            return json.loads(result_text)

        except errors.ServerError as e:
            if e.code != 503:
                raise
            if attempt == 2:
                raise

            time.sleep(5 * (2 ** attempt))


In [111]:

logic_review = review_financial_logic(
    financial_analysis,
    financial_metrics
)

print(json.dumps(
    logic_review,
    ensure_ascii=False,
    indent=2
))


{
  "issues": [
    {
      "original": "높은 수준의 영업이익률(48.5928%) [M1]과 ROE(44.1512%) [M5]를 기록하였으며, 부채비율은 45.9455% [M4], 영업현금흐름 성장률은 79.1292% [M6]를 나타내 재무 및 손익 전반의 수치가 전년 대비 개선된 양상을 보였습니다.",
      "status": "REVIEW",
      "reason": "제공된 데이터 중 전년 대비 비교가 가능한 지표는 성장률 지표(매출, 영업이익, 영업현금흐름)뿐이며, 영업이익률·ROE·부채비율은 전년도 수치가 제공되지 않아 전년 대비 개선 여부를 확인할 수 없음에도 '재무 및 손익 전반의 수치가 전년 대비 개선되었다'고 단정했습니다.",
      "suggestion": "높은 수준의 영업이익률(48.5928%) [M1]과 ROE(44.1512%) [M5], 50% 미만의 부채비율(45.9455%) [M4]을 기록한 가운데, 매출·영업이익·영업현금흐름 등 주요 성장성 및 현금흐름 지표가 전년 대비 큰 폭으로 증가했습니다."
    }
  ],
  "overall": "REVIEW"
}


In [112]:

if logic_review["issues"]:
    logic_review_table = pd.DataFrame(
        logic_review["issues"]
    )
    display(logic_review_table)
else:
    print("Gemini가 지적한 논리적 문제가 없습니다.")

print("AI 논리 검토 의견:", logic_review["overall"])


,original,status,reason,suggestion
0,높은 수준의 영업이익률(48.5928%) [M1]과 ROE(44.1512%) [M5...,REVIEW,"제공된 데이터 중 전년 대비 비교가 가능한 지표는 성장률 지표(매출, 영업이익, 영...",높은 수준의 영업이익률(48.5928%) [M1]과 ROE(44.1512%) [M5...


AI 논리 검토 의견: REVIEW


In [113]:

# 1. Python 계산 검증
calculation_ok = (
    len(validation_table) == 6
    and (validation_table["판정"] == "PASS").all()
)

# 2. 원본 데이터 연결 검증
sources_ok = (
    len(source_validation) == 6
    and (source_validation["RID검사"] == "PASS").all()
    and (source_validation["출처검사"] == "PASS").all()
)

# 3. 보고서의 M# 존재 여부
references_ok = report_check["status"] == "PASS"

# 4. 보고서의 숫자 검사 결과
number_statuses = set(number_validation["판정"])

if "FIX" in number_statuses:
    numbers_status = "FIX"
elif "REVIEW" in number_statuses:
    numbers_status = "REVIEW"
elif number_statuses == {"PASS"}:
    numbers_status = "PASS"
else:
    numbers_status = "REVIEW"

# 5. Gemini 논리 검토 의견
logic_status = logic_review["overall"]

# 통합 검증표
final_validation = pd.DataFrame([
    {
        "검증 항목": "재무지표 계산",
        "판정": "PASS" if calculation_ok else "FIX"
    },
    {
        "검증 항목": "원본 RID 및 출처 연결",
        "판정": "PASS" if sources_ok else "FIX"
    },
    {
        "검증 항목": "보고서 M# 존재 여부",
        "판정": "PASS" if references_ok else "FIX"
    },
    {
        "검증 항목": "보고서 백분율 대조",
        "판정": numbers_status
    },
    {
        "검증 항목": "AI 논리 검토 의견",
        "판정": logic_status
    }
])

display(final_validation)


,검증 항목,판정
0,재무지표 계산,PASS
1,원본 RID 및 출처 연결,PASS
2,보고서 M# 존재 여부,PASS
3,보고서 백분율 대조,REVIEW
4,AI 논리 검토 의견,REVIEW


In [114]:

statuses = set(final_validation["판정"])

if "FIX" in statuses:
    overall_status = "FIX"
elif "REVIEW" in statuses:
    overall_status = "REVIEW"
elif statuses == {"PASS"}:
    overall_status = "PASS"
else:
    overall_status = "REVIEW"

print("종합 판정:", overall_status)


종합 판정: REVIEW


In [115]:

research_package = {
    "company": "SK하이닉스",
    "industry_analysis": industry_result["analysis"],
    "financial_metrics": metric_payload,
    "valuation": valuation_result["scenarios"],
    "valuation_assumptions": "학습용 임의 가정",
    "validation": final_validation.to_dict("records"),
    "overall_status": overall_status,
    "logic_issues": logic_review["issues"]
}

print("기업:", research_package["company"])
print("재무지표:", len(research_package["financial_metrics"]))
print("밸류에이션 시나리오:", len(research_package["valuation"]))
print("종합 검증 상태:", research_package["overall_status"])


기업: SK하이닉스
재무지표: 6
밸류에이션 시나리오: 3
종합 검증 상태: REVIEW


In [116]:

def generate_research_report(package):

    prompt = f"""
너는 금융 리서치 통합 보고서 작성 담당 AI다.

[분석 데이터]
{json.dumps(package, ensure_ascii=False, indent=2, default=str)}

다음 목차로 한국어 보고서를 작성하라.

1. Executive Summary
2. 산업분석
3. 기업 재무분석
4. 밸류에이션 시나리오
5. 주요 위험 요인
6. 검증 결과 및 한계
7. 추가 조사 과제

[필수 규칙]
- 제공된 데이터만 사용한다.
- 산업 사실에는 [I#], 재무지표에는 [M#]를 표시한다.
- 출처가 있다는 사실과 원문 검증 완료를 혼동하지 않는다.
- 논리 검토에서 지적된 원문을 그대로 반복하지 않는다.
- 전년도 비교가 없는 지표의 개선을 단정하지 않는다.
- 밸류에이션의 EPS 변화율과 PER은 임의 가정임을 명시한다.
- 시나리오 가격을 적정주가나 목표주가라고 부르지 않는다.
- REVIEW 항목은 검토 필요 사항으로 표시한다.
- 자료에 없는 현재 주가, 시장점유율, 미래 실적을 만들어내지 않는다.
- 산업·재무·밸류에이션 사이의 관계를 설명하되
  근거 없는 인과관계를 만들지 않는다.
- 본 보고서가 검증 완료된 투자보고서는 아님을 명시한다.
"""

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt
            )

            if not response.text:
                raise ValueError("보고서 응답이 비어 있습니다.")

            return response.text

        except errors.ServerError as e:
            if e.code != 503:
                raise
            if attempt == 2:
                raise
            time.sleep(5 * (2 ** attempt))


In [117]:

final_report = generate_research_report(research_package)

print(final_report)


# [기업 및 산업 분석 통합 보고서: SK하이닉스]

---

## 1. Executive Summary

* **산업 환경**: 2026년 상반기 세계 반도체 시장이 전년 동기 대비 102% 성장한 가운데, 특히 메모리 부문은 305% 급성장하였습니다 [I1]. 이러한 확장의 핵심 동력은 AI 인프라 투자이며, 일반 서버용 메모리 수요 역시 함께 증가하고 있습니다 [I1, I2].
* **기업 실적 및 재무**: SK하이닉스는 2025년 HBM(고대역폭 메모리) 매출이 전년 대비 2배 이상 증가했습니다 [I2]. 2025년 기준 매출 성장률 46.7629% [M2], 영업이익 성장률 101.1577% [M3], 영업현금흐름 성장률 79.1292% [M6]를 기록하며 높은 성장세를 나타냈습니다. 또한 영업이익률 48.5928% [M1], ROE 44.1512% [M5], 부채비율 45.9455% [M4]를 기록했습니다.
* **밸류에이션 및 유의 사항**: 학습용 임의 가정(PER 5~9배, EPS 변화율 -30%~+30%)을 적용한 시나리오 주가는 비관 217,154원부터 낙관 725,915원까지 산출되었습니다. 본 보고서에 포함된 데이터 및 논리 검토 항목 중 일부가 '검토 필요(REVIEW)' 판정을 받았으므로, 본 보고서는 검증이 최종 완료된 투자보고서가 아님을 명시합니다.

---

## 2. 산업분석

### (1) 반도체 시장 성장 동력
* **관측된 사실**: 2026년 상반기 전 세계 반도체 시장은 전년 동기 대비 102% 성장률을 기록했으며, 메모리 부문 성장률은 305%에 달했습니다 [I1]. 해당 성장의 주요 동인은 'AI 인프라 투자'입니다 [I1]. 또한 AI 인프라 외에 서버용 일반 메모리 수요 역시 증가세를 보였습니다 [I2].
* **산업 추론**: 반도체 전체 시장 성장률(102%) 대비 메모리 부문의 성장률(305%)이 현저히 높다는 점에서, 현재 반도체 사이클은 AI 인프라 확충에 필수적인 메모리 수요가 주도하고 있는 것으

In [118]:

from pathlib import Path

report_path = Path("/content/SK하이닉스_통합리포트.md")

report_path.write_text(
    final_report,
    encoding="utf-8"
)

print("저장 완료:", report_path)


저장 완료: /content/SK하이닉스_통합리포트.md


In [120]:
from google.colab import files

files.download("/content/SK하이닉스_통합리포트.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>